In [13]:
#Set up the environment and imports
import os
import sys
from dotenv import load_dotenv
from langfuse.openai import openai
from langfuse import observe

# Add the repo root to path so we can import from shared/
sys.path.append(os.path.abspath(".."))
from shared.shopbot_config import SHOPBOT_SYSTEM_PROMPT

load_dotenv("../.env")

client = openai.OpenAI()

MODEL = "gpt-4o-mini"

In [14]:
POLICY_SNIPPETS = {
    "shipping": "Free shipping is available on all orders over $75.",
    "loyalty": "The StyleHive Rewards loyalty program is open to all registered members.",
    "support hours": "Customer support is available Monday to Friday, 9am-6pm EST.",
    "30-day return": "StyleHive accepts returns within 30 days of delivery for unworn items with tags attached.",
}

#Decorating the retrieval step so it shows up as its own span
@observe()
def retrieve_policy(question: str) -> str:
    for keyword, snippet in POLICY_SNIPPETS.items():
        if keyword in question.lower():
            return snippet
    return "No specific policy found."

In [15]:
#Decorating the generation step so it shows up as its own span
@observe()
def generate_response(question: str, context: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SHOPBOT_SYSTEM_PROMPT},
            {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"},
        ],
    )
    return response.choices[0].message.content

In [16]:
#Setting trace name to "ask-shopbot-pipeline" and observing the pipeline
@observe(name="ask-shopbot-pipeline")
def ask_shopbot_pipeline(question: str) -> str:
    context = retrieve_policy(question)
    answer = generate_response(question, context)
    return answer

In [17]:
#interacting with the pipeline and observing the function
ask_shopbot_pipeline("Do you accept returns on Black Friday sale items?")

"I don't have information about specific policies for Black Friday sale items. However, StyleHive generally has a 30-day return policy for unworn items with tags attached. You may want to check the terms during the sale or contact customer support for more details."